In [18]:
import os
os.chdir('/Users/burkelawlor/Repos/n23-lrrk2-biomarkers')

from pathlib import Path
from glob import glob
import pandas as pd
import numpy as np

from utils.data_processing import append_to_cleaned_biospecimen_csv, _CANONICAL_CLEANED_BIOSPECIMEN_COLUMNS

DATA_DIR = Path("./data/raw")

In [19]:
ml_df_full = pd.read_csv(DATA_DIR / "AMPPDv4_LRRK2v4_results_N23.csv")
ml_df_posthoc = pd.read_csv(DATA_DIR / "AMPPDv4_LRRK2v4_results_N23_for_post_hoc.csv")

ml_df_full['GBA'] = (~ml_df_full.ID.isin(ml_df_posthoc.ID)).astype(int)

ml_df = ml_df_full[ml_df_full.ID.str.contains('PP-')]
ml_df['PATNO'] = ml_df.ID.str.strip('PP-').astype(int)
ml_df.rename(columns={'LRRK2-RV': 'RV', 'LRRK2-Predicted': 'PREDICTED', 'LRRK2-Driven': 'DRIVEN', 'heuristic': 'HEURISTIC'}, inplace=True)
ml_df = ml_df[['PATNO', 'RV', 'GBA', 'PREDICTED', 'DRIVEN', 'HEURISTIC']]

In [20]:
age = pd.read_csv(DATA_DIR / 'Age_at_visit_24Mar2026.csv').rename(columns={'EVENT_ID': 'CLINICAL_EVENT'})
age = age.drop_duplicates(subset=['PATNO', 'CLINICAL_EVENT'], keep='last')

In [21]:
biomarker_df = pd.read_csv(DATA_DIR / "Current_Biospecimen_Analysis_Results_06Mar2026.csv")
print(len(biomarker_df))
biomarker_df.drop_duplicates(subset=['PATNO','PROJECTID','CLINICAL_EVENT','TYPE','TESTNAME','RUNDATE'], keep='last', inplace=True)
print(len(biomarker_df))

biomarker_df = biomarker_df.merge(ml_df, on='PATNO', how='left')
print(len(biomarker_df))

biomarker_df = biomarker_df.merge(age, on=['PATNO', 'CLINICAL_EVENT'], how='left')
print(len(biomarker_df))

/var/folders/l8/r_bqmkfd36n6y9mvy600d2580000gn/T/ipykernel_97283/1316675356.py:1: DtypeWarning: Columns (0: TESTVALUE, 1: UNITS) have mixed types. Specify dtype option on import or set low_memory=False.
  biomarker_df = pd.read_csv(DATA_DIR / "Current_Biospecimen_Analysis_Results_06Mar2026.csv")


1010327
619531
619531
619531


# Project 145 (Urine BMP)

In [22]:
project_145 = biomarker_df[biomarker_df.PROJECTID == 145]
project_145.TESTNAME.unique()

<StringArray>
['2,2' di-22:6-BMP', 'total di-18:1-BMP', 'total di-22:6-BMP']
Length: 3, dtype: str

In [23]:
project_145[list(_CANONICAL_CLEANED_BIOSPECIMEN_COLUMNS)]

,PATNO,SEX,AGE_AT_VISIT,COHORT,CLINICAL_EVENT,TYPE,TESTNAME,TESTVALUE,UNITS,RUNDATE,PROJECTID,RV,GBA,PREDICTED,DRIVEN,HEURISTIC
298255,3000,Female,69.1,Control,BL,Urine,"2,2' di-22:6-BMP",4.51,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
298256,3000,Female,69.1,Control,BL,Urine,total di-18:1-BMP,0.65,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
298257,3000,Female,69.1,Control,BL,Urine,total di-22:6-BMP,6.53,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
298258,3001,Male,65.1,PD,BL,Urine,"2,2' di-22:6-BMP",4.34,ng/mg Creatinine,2019-07-29,145,0.0,0.0,0.0,0.0,Non
298259,3001,Male,65.1,PD,BL,Urine,total di-18:1-BMP,2.79,ng/mg Creatinine,2019-07-29,145,0.0,0.0,0.0,0.0,Non
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303395,75565,Male,65.6,Prodromal,BL,Urine,total di-18:1-BMP,0.73,ng/mg Creatinine,2019-08-01,145,0.0,1.0,1.0,1.0,Predicted
303396,75565,Male,65.6,Prodromal,BL,Urine,total di-22:6-BMP,4.30,ng/mg Creatinine,2019-08-01,145,0.0,1.0,1.0,1.0,Predicted
303397,75570,Male,51.2,Prodromal,BL,Urine,"2,2' di-22:6-BMP",8.81,ng/mg Creatinine,2019-07-19,145,NaN,NaN,NaN,NaN,NaN
303398,75570,Male,51.2,Prodromal,BL,Urine,total di-18:1-BMP,7.29,ng/mg Creatinine,2019-07-19,145,NaN,NaN,NaN,NaN,NaN


In [24]:
append_to_cleaned_biospecimen_csv(project_145)
# project_145.to_csv('../data/processed/test.csv', index=False)

,PATNO,SEX,AGE_AT_VISIT,COHORT,CLINICAL_EVENT,TYPE,TESTNAME,TESTVALUE,UNITS,RUNDATE,PROJECTID,RV,GBA,PREDICTED,DRIVEN,HEURISTIC
0,3000,Female,69.1,Control,BL,Urine,"2,2' di-22:6-BMP",4.51,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
1,3000,Female,69.1,Control,BL,Urine,total di-18:1-BMP,0.65,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
2,3000,Female,69.1,Control,BL,Urine,total di-22:6-BMP,6.53,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
3,3001,Male,65.1,PD,BL,Urine,"2,2' di-22:6-BMP",4.34,ng/mg Creatinine,2019-07-29,145,0.0,0.0,0.0,0.0,Non
4,3001,Male,65.1,PD,BL,Urine,total di-18:1-BMP,2.79,ng/mg Creatinine,2019-07-29,145,0.0,0.0,0.0,0.0,Non
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5140,75565,Male,65.6,Prodromal,BL,Urine,total di-18:1-BMP,0.73,ng/mg Creatinine,2019-08-01,145,0.0,1.0,1.0,1.0,Predicted
5141,75565,Male,65.6,Prodromal,BL,Urine,total di-22:6-BMP,4.30,ng/mg Creatinine,2019-08-01,145,0.0,1.0,1.0,1.0,Predicted
5142,75570,Male,51.2,Prodromal,BL,Urine,"2,2' di-22:6-BMP",8.81,ng/mg Creatinine,2019-07-19,145,NaN,NaN,NaN,NaN,NaN
5143,75570,Male,51.2,Prodromal,BL,Urine,total di-18:1-BMP,7.29,ng/mg Creatinine,2019-07-19,145,NaN,NaN,NaN,NaN,NaN


# Project 151 (CSF SomaScan Proteomics)

In [25]:
# project_151 = pd.read_csv('./data/raw/Project_151_pQTL_in_CSF_1_of_7_Batch_Corrected__24Mar2026.csv')

files = glob('./data/raw/Project_151_pQTL_in_CSF_*_of_7_Batch_Corrected__24Mar2026.csv')
project_151 = pd.concat([pd.read_csv(f) for f in files])
print(len(project_151))

5541030


In [26]:
key = pd.read_csv('./data/raw/PPMI_Project_151_pqtl_Analysis_Annotations_20210210.csv', usecols=['SOMA_SEQ_ID', 'TARGET_GENE_SYMBOL'])
key['TESTNAME_2'] =  key['TARGET_GENE_SYMBOL'] + '_' + key['SOMA_SEQ_ID']
key['TESTNAME_2'] = key['TESTNAME_2'].fillna(key['SOMA_SEQ_ID'])
key.drop_duplicates(inplace=True)

project_151 = project_151.merge(key, left_on='TESTNAME', right_on=['SOMA_SEQ_ID'], how='left')
project_151['TESTNAME'] = project_151['TESTNAME_2']
print(len(project_151))

5660304


In [27]:
project_151 = project_151.merge(ml_df, on='PATNO', how='left')
print(len(project_151))

project_151 = project_151.merge(age, on=['PATNO', 'CLINICAL_EVENT'], how='left')
print(len(project_151))

5660304
5660304


In [28]:
ml_df

,PATNO,RV,GBA,PREDICTED,DRIVEN,HEURISTIC
8023,10874,0,0,1,1,Predicted
8024,12224,0,0,0,0,Non
8025,12499,0,0,0,0,Non
8026,12593,0,0,0,0,Non
8027,13039,0,0,0,0,Non
...,...,...,...,...,...,...
9825,90456,0,0,1,1,Predicted
9826,91097,0,0,1,1,Predicted
9827,91837,0,0,1,1,Predicted
9828,92490,0,0,0,0,Non


In [29]:
project_151

,PATNO,SEX,COHORT,CLINICAL_EVENT,TYPE,TESTNAME,TESTVALUE,UNITS,PLATEID,RUNDATE,...,update_stamp,SOMA_SEQ_ID,TARGET_GENE_SYMBOL,TESTNAME_2,RV,GBA,PREDICTED,DRIVEN,HEURISTIC,AGE_AT_VISIT
0,56807,Female,Prodromal,BL,Cerebrospinal Fluid,MFAP1_5606-24_3,6.045059,log2 RFU,P0024858,2019-02-12,...,2022-05-13 10:38:41,5606-24_3,MFAP1,MFAP1_5606-24_3,0.0,1.0,1.0,1.0,Predicted,51.5
1,56807,Female,Prodromal,BL,Cerebrospinal Fluid,MFNG_5605-77_3,7.326083,log2 RFU,P0024858,2019-02-12,...,2022-05-13 10:38:41,5605-77_3,MFNG,MFNG_5605-77_3,0.0,1.0,1.0,1.0,Predicted,51.5
2,56807,Female,Prodromal,BL,Cerebrospinal Fluid,HPSE_5604-30_3,6.347686,log2 RFU,P0024858,2019-02-12,...,2022-05-13 10:38:41,5604-30_3,HPSE,HPSE_5604-30_3,0.0,1.0,1.0,1.0,Predicted,51.5
3,56807,Female,Prodromal,BL,Cerebrospinal Fluid,RNASE10_5602-62_3,6.448036,log2 RFU,P0024858,2019-02-12,...,2022-05-13 10:38:41,5602-62_3,RNASE10,RNASE10_5602-62_3,0.0,1.0,1.0,1.0,Predicted,51.5
4,56807,Female,Prodromal,BL,Cerebrospinal Fluid,PGLYRP2_5601-2_3,10.145977,log2 RFU,P0024858,2019-02-12,...,2022-05-13 10:38:41,5601-2_3,PGLYRP2,PGLYRP2_5601-2_3,0.0,1.0,1.0,1.0,Predicted,51.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5660299,4020,Male,PD,BL,Cerebrospinal Fluid,APOD_4712-28_2,8.357636,log2 RFU,P0024866,2019-02-12,...,2022-05-13 10:38:32,4712-28_2,APOD,APOD_4712-28_2,0.0,0.0,0.0,0.0,Non,60.8
5660300,4020,Male,PD,BL,Cerebrospinal Fluid,IL3_4717-55_2,7.960881,log2 RFU,P0024866,2019-02-12,...,2022-05-13 10:38:32,4717-55_2,IL3,IL3_4717-55_2,0.0,0.0,0.0,0.0,Non,60.8
5660301,4020,Male,PD,BL,Cerebrospinal Fluid,PPIB_4718-5_2,7.402274,log2 RFU,P0024866,2019-02-12,...,2022-05-13 10:38:32,4718-5_2,PPIB,PPIB_4718-5_2,0.0,0.0,0.0,0.0,Non,60.8
5660302,4020,Male,PD,BL,Cerebrospinal Fluid,PDIA3_4719-58_2,8.859675,log2 RFU,P0024866,2019-02-12,...,2022-05-13 10:38:32,4719-58_2,PDIA3,PDIA3_4719-58_2,0.0,0.0,0.0,0.0,Non,60.8


In [30]:
project_151.TESTNAME.nunique()

4888

In [31]:
append_to_cleaned_biospecimen_csv(project_151)

,PATNO,SEX,AGE_AT_VISIT,COHORT,CLINICAL_EVENT,TYPE,TESTNAME,TESTVALUE,UNITS,RUNDATE,PROJECTID,RV,GBA,PREDICTED,DRIVEN,HEURISTIC
0,3000,Female,69.1,Control,BL,Urine,"2,2' di-22:6-BMP",4.510000,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
1,3000,Female,69.1,Control,BL,Urine,total di-18:1-BMP,0.650000,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
2,3000,Female,69.1,Control,BL,Urine,total di-22:6-BMP,6.530000,ng/mg Creatinine,2019-07-22,145,0.0,0.0,1.0,1.0,Predicted
3,3001,Male,65.1,PD,BL,Urine,"2,2' di-22:6-BMP",4.340000,ng/mg Creatinine,2019-07-29,145,0.0,0.0,0.0,0.0,Non
4,3001,Male,65.1,PD,BL,Urine,total di-18:1-BMP,2.790000,ng/mg Creatinine,2019-07-29,145,0.0,0.0,0.0,0.0,Non
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5665444,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,APOD_4712-28_2,8.357636,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
5665445,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,IL3_4717-55_2,7.960881,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
5665446,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,PPIB_4718-5_2,7.402274,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
5665447,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,PDIA3_4719-58_2,8.859675,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non


In [32]:
project_151[list(_CANONICAL_CLEANED_BIOSPECIMEN_COLUMNS)]

,PATNO,SEX,AGE_AT_VISIT,COHORT,CLINICAL_EVENT,TYPE,TESTNAME,TESTVALUE,UNITS,RUNDATE,PROJECTID,RV,GBA,PREDICTED,DRIVEN,HEURISTIC
0,56807,Female,51.5,Prodromal,BL,Cerebrospinal Fluid,MFAP1_5606-24_3,6.045059,log2 RFU,2019-02-12,151,0.0,1.0,1.0,1.0,Predicted
1,56807,Female,51.5,Prodromal,BL,Cerebrospinal Fluid,MFNG_5605-77_3,7.326083,log2 RFU,2019-02-12,151,0.0,1.0,1.0,1.0,Predicted
2,56807,Female,51.5,Prodromal,BL,Cerebrospinal Fluid,HPSE_5604-30_3,6.347686,log2 RFU,2019-02-12,151,0.0,1.0,1.0,1.0,Predicted
3,56807,Female,51.5,Prodromal,BL,Cerebrospinal Fluid,RNASE10_5602-62_3,6.448036,log2 RFU,2019-02-12,151,0.0,1.0,1.0,1.0,Predicted
4,56807,Female,51.5,Prodromal,BL,Cerebrospinal Fluid,PGLYRP2_5601-2_3,10.145977,log2 RFU,2019-02-12,151,0.0,1.0,1.0,1.0,Predicted
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5660299,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,APOD_4712-28_2,8.357636,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
5660300,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,IL3_4717-55_2,7.960881,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
5660301,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,PPIB_4718-5_2,7.402274,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
5660302,4020,Male,60.8,PD,BL,Cerebrospinal Fluid,PDIA3_4719-58_2,8.859675,log2 RFU,2019-02-12,151,0.0,0.0,0.0,0.0,Non
